In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch as t
from transformer_lens import HookedTransformer
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from dadapy.data import Data
from collections import defaultdict


import transformer_lens.utils as utils
import einops

from joblib import Parallel, delayed
import pandas as pd

import plot_utils

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
# Saves computation time, since we don't need it for the contents of this notebook
t.set_grad_enabled(False)


In [3]:
# Load GPT-2 Small
model = HookedTransformer.from_pretrained("gpt2-small")

# Load prompt from Pile-10K (Prompt 3218)
pile_dataset = load_dataset("NeelNanda/pile-10k")

Loaded pretrained model gpt2-small into HookedTransformer


In [4]:
filtered_indices = np.load('filtered_indices.npy')


In [5]:
filtered_dataset = pile_dataset['train'][filtered_indices]

In [6]:
setname_to_indexlist = defaultdict(list)

In [7]:
for i, set_name in enumerate(filtered_dataset['meta']):
    setname_to_indexlist[set_name['pile_set_name']].append(i)

In [8]:
arxiv_indices, wiki_indices, math_indices = setname_to_indexlist['ArXiv'], setname_to_indexlist['Wikipedia (en)'], setname_to_indexlist['DM Mathematics']

In [9]:
arxiv_prompts, wiki_prompts, math_prompts = [filtered_dataset['text'][idx] for idx in arxiv_indices], [filtered_dataset['text'][idx] for idx in wiki_indices], [filtered_dataset['text'][idx] for idx in math_indices]


In [10]:
def prompts_to_tokens(prompts):
    tokens = model.to_tokens(prompts, prepend_bos=True)
    tokens = tokens[..., :512]
    return tokens

In [11]:
arxiv_tokens, wiki_tokens, math_tokens = prompts_to_tokens(arxiv_prompts), prompts_to_tokens(wiki_prompts), prompts_to_tokens(math_prompts)

In [12]:
# Function to compute intrinsic dimension (ID) with dadapy
def compute_ids(full_reps):
    ids = []
    for rep in full_reps:
        _data = Data(coordinates=rep.cpu().detach().numpy(), maxk=100)
        ids.append(_data.return_id_scaling_gride(range_max=64))
    return np.array(ids)

In [13]:
def zero_attn_out_hook(attn_out, hook):
    # print(attn_out.shape)
    return t.zeros_like(attn_out)

zero_attn_hooks = [
            # Changed hook point to blocks.{layer}.attn.hook_result
            (utils.get_act_name("attn_out", layer), zero_attn_out_hook)
            for layer in range(model.cfg.n_layers)
        ]

In [14]:
def tokens_to_idim_before_after_layerNorm(tokens, N):
    logits, cache = model.run_with_cache(tokens[:N])
    accumulated_residual, labels = cache.accumulated_resid(layer = -1, incl_mid=True,\
                                                return_labels = True)
    
    
    
    print(accumulated_residual.shape)
    

    scaled_residual_stack = cache.apply_ln_to_stack(
        accumulated_residual, layer=-1, pos_slice=-1
    )
    
    print(scaled_residual_stack.shape)
    

        
    idim_before = []
    idim_after = []
    
    
    for idx in range(len(arxiv_tokens[:N])):
        ids = compute_ids(accumulated_residual[:, idx])
        ids_after = compute_ids(scaled_residual_stack[:, idx])
        idim_before.append(ids[:, 0, 1])
        idim_after.append(ids_after[:, 0, 1])
    # print(accumulated_residual.shape)
    return labels, idim_before, idim_after
        

In [15]:
def get_interaction_values(idim, idim_free):
    idim_interaction = np.array(idim_free) - np.array(idim)
    g_interaction = idim_interaction / np.array(idim)
    return idim_interaction, g_interaction

In [16]:
labels, idim_before, idim_after = tokens_to_idim_before_after_layerNorm(arxiv_tokens, 1)

torch.Size([25, 1, 512, 768])
torch.Size([25, 1, 512, 768])


In [31]:
idim_before[0][0]

2.5424704483599934

In [32]:
idim_after[0][0]

2.5424704483599934

In [27]:
idim_before[0] == idim_after[0]

array([ True,  True,  True,  True,  True,  True,  True,  True, False,
        True,  True, False,  True,  True,  True, False,  True,  True,
        True,  True, False,  True,  True, False, False])

In [33]:
idim_after[0][-1], idim_before[0][-1]

(6.554799315965298, 6.554799257757695)

In [17]:
def summary_statistics(idim, n_bootstrap=1000):
    y_data = np.array(idim)
    
    num_lines, num_points = y_data.shape
    # --- 2. Calculate Average and Bootstrap Standard Error ---

    # Calculate the average y-value for each x-point across all 10 lines.
    y_average = np.mean(y_data, axis=0)

    # Calculate error bars using bootstrapping
    bootstrap_std_err = np.zeros(num_points)

    for j in range(num_points): # For each x-point (column)
        point_data = y_data[:, j] # Get the y-values from all lines for this x-point
        bootstrap_means = np.zeros(n_bootstrap)

        for i in range(n_bootstrap):
            # Resample the data *with replacement*
            resampled_data = np.random.choice(point_data, size=num_lines, replace=True)
            # Calculate the mean of the resampled data
            bootstrap_means[i] = np.mean(resampled_data)

        # Calculate the standard deviation of the bootstrap means
        # This serves as the bootstrapped estimate of the standard error.
        # Alternatively, one could calculate a confidence interval (e.g., 2.5th and 97.5th percentiles)
        # from bootstrap_means for potentially asymmetric error bars.
        bootstrap_std_err[j] = np.std(bootstrap_means, ddof=1)
    return y_average, bootstrap_std_err
        

In [18]:
arxiv_before_average, arxiv_before_err = summary_statistics(idim_before)
arxiv_after_average, arxiv_after_err = summary_statistics(idim_after)

In [19]:
arxiv_before_average

array([2.54, 3.47, 2.75, 3.15, 2.9 , 3.54, 3.5 , 3.84, 3.78, 4.18, 4.1 ,
       4.83, 4.73, 5.34, 5.13, 5.73, 5.64, 6.  , 5.97, 6.13, 6.27, 6.19,
       6.38, 6.33, 6.55])

In [20]:
arxiv_after_average

array([2.54, 3.47, 2.75, 3.15, 2.9 , 3.54, 3.5 , 3.84, 3.78, 4.18, 4.1 ,
       4.83, 4.73, 5.34, 5.13, 5.73, 5.64, 6.  , 5.97, 6.13, 6.27, 6.19,
       6.38, 6.33, 6.55])

In [21]:
arxiv_before_err

array([4.44e-16, 0.00e+00, 8.89e-16, 8.89e-16, 1.33e-15, 4.44e-16,
       8.89e-16, 4.44e-16, 0.00e+00, 0.00e+00, 8.89e-16, 8.89e-16,
       8.89e-16, 2.67e-15, 8.89e-16, 1.78e-15, 0.00e+00, 8.89e-16,
       8.89e-16, 2.67e-15, 1.78e-15, 2.67e-15, 1.78e-15, 8.89e-16,
       8.89e-16])

In [22]:
import plotly.graph_objects as go
import plotly
from plotly.colors import qualitative
colors = qualitative.Plotly # Get the default sequence
D3colors = qualitative.D3

In [23]:
def get_alpha_color_string(hex_color, alpha = 0.5):
    rgb_tuple = plotly.colors.hex_to_rgb(hex_color)
    return f'rgba({rgb_tuple[0]}, {rgb_tuple[1]}, {rgb_tuple[2]}, {alpha})'

In [24]:
# Initialize a Plotly Figure object
fig = go.Figure()



# Example: Get the first color


# Add the main trace: the average line with error bars
fig.add_trace(go.Scatter(
    x=labels,
    y=arxiv_before_average,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Arxiv Idim Prompt Before LayerNorm',   # Name for the legend
    line=dict(color=colors[0], width=2), # Style the average line
    marker=dict(size=5, color=colors[0]), # Style the markers
    error_y=dict(
        type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
        array=arxiv_before_err,   # Provide the calculated standard error values for the error bars
        visible=True,      # Make the error bars visible
        thickness=1,       # Thickness of the error bar lines
        width=3,           # Width of the e rror bar caps
        color=get_alpha_color_string(colors[0]) # Color of error bars (royalblue with some transparency)
    )
))



fig.add_trace(go.Scatter(
    x=labels,
    y=arxiv_after_average,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Arxiv Idim Prompt After LayerNorm',   # Name for the legend
    line=dict(color=colors[1], width=2), # Style the average line
    marker=dict(size=5, color=colors[1]), # Style the markers
    error_y=dict(
        type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
        array=arxiv_after_err,   # Provide the calculated standard error values for the error bars
        visible=True,      # Make the error bars visible
        thickness=1,       # Thickness of the error bar lines
        width=3,           # Width of the e rror bar caps
        color=get_alpha_color_string(colors[1]) # Color of error bars (royalblue with some transparency)
    )
))


# --- Optional: Add the original individual lines for context (commented out by default) ---
# Uncomment the following loop if you want to see the underlying data lines.
# for i in range(num_lines):
#     fig.add_trace(go.Scatter(
#         x=x_values,
#         y=y_data[i, :],
#         mode='lines',
#         name=f'Line {i+1}',
#         line=dict(width=0.5, color='rgba(128, 128, 128, 0.4)'), # Thin, semi-transparent grey lines
#         showlegend=False # Hide these individual lines from the main legend
#     ))

# --- 4. Customize the Plot Layout ---
fig.update_layout(font=dict(size=22)) 
fig.update_layout(
    title='Arxiv, Before After LayerNorm', # Plot title
    xaxis_title='Layers',                 # X-axis label
    yaxis_title='IDim',                               # Y-axis label
    legend_title='LayerNorm Applied',                               # Title for the legend box
    hovermode='x unified',                               # Show hover info for all traces at a given x-value
    template='plotly_white',
    showlegend=True,
)

#fig.write_image('IDIM_prompt_categories.png')
# --- 5. Show the Plot ---

# Display the figure. This will typically open it in a browser window
# or display it in the output cell of a Jupyter notebook.
fig.show()
